# IndicConformer - 22 Language ASR Test

Audio -> IndicConformer -> Indian language text

One multilingual pretrained model transcribes all 22 scheduled languages.

Model: `ai4bharat/indic-conformer-600m-multilingual`

Audio files: `ai/speech/audio/<language>_test.wav`

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import time
from pathlib import Path

import soundfile as sf
import torchaudio
from transformers import AutoModel


# 22 scheduled languages: name -> IndicConformer language code
LANGUAGES = {
    "Assamese": "as", "Bengali": "bn", "Bodo": "brx", "Dogri": "doi",
    "Gujarati": "gu", "Hindi": "hi", "Kannada": "kn", "Kashmiri": "ks",
    "Konkani": "kok", "Maithili": "mai", "Malayalam": "ml", "Manipuri": "mni",
    "Marathi": "mr", "Nepali": "ne", "Odia": "or", "Punjabi": "pa",
    "Sanskrit": "sa", "Santali": "sat", "Sindhi": "sd", "Tamil": "ta",
    "Telugu": "te", "Urdu": "ur",
}

MODEL_ID = "ai4bharat/indic-conformer-600m-multilingual"
AUDIO_DIR = Path("audio")  # run from ai/speech/ so this notebook finds the audio folder
SAMPLE_RATE = 16000

e:\VaaniDoc\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the ONE multilingual model, used for all languages
print("Loading IndicConformer ...")
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(device)
model.eval()
print("Model loaded on", device)

Loading IndicConformer ...
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files: 100%|██████████| 404/404 [00:00<00:00, 1171.26it/s]


Please check FRAME_DURATION_MS. The timestamps can be inaccurate


e:\VaaniDoc\.venv\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Model loaded on cuda


In [5]:
def load_audio(path):
    """Read audio with soundfile, return tensor (channels, samples) at 16 kHz."""
    if path.stat().st_size == 0:
        raise FileNotFoundError(f"Empty file: {path}")
    data, sr = sf.read(str(path), dtype="float32")
    if data.ndim > 1:
        data = data.mean(axis=1)  # convert stereo to mono
    wav = torch.from_numpy(data[None, :])
    if sr != SAMPLE_RATE:  # resample to 16 kHz if needed
        wav = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(wav)
    return wav

In [6]:
results = []  # [language, code, transcript, time, status]

for lang, code in LANGUAGES.items():
    path = AUDIO_DIR / f"{lang}_test.wav"
    t0 = time.time()

    try:
        if not path.exists():
            raise FileNotFoundError(path)
        wav = load_audio(path).to(device)
        text = model(wav, code, "ctc")
        secs = round(time.time() - t0, 2)
        results.append([lang, code, text, secs, "OK"])
    except FileNotFoundError:
        secs = round(time.time() - t0, 2)
        results.append([lang, code, "Audio not available", secs, "Audio not available"])
    except Exception as e:
        secs = round(time.time() - t0, 2)
        results.append([lang, code, "Audio not available", secs, f"Error: {type(e).__name__}"])
        print(f"{lang:12s} | {code:4s} | {type(e).__name__}: {e}")

In [8]:
print("Language | Code | Transcript | Time | Status")
print("-" * 80)
for lang, code, text, secs, status in results:
    print(f"{lang} | {code} | {text} | {secs}s | {status}")

Language | Code | Transcript | Time | Status
--------------------------------------------------------------------------------
Assamese | as | Audio not available | 0.0s | Audio not available
Bengali | bn | Audio not available | 0.0s | Audio not available
Bodo | brx | Audio not available | 0.0s | Audio not available
Dogri | doi | Audio not available | 0.0s | Audio not available
Gujarati | gu | Audio not available | 0.0s | Audio not available
Hindi | hi | Audio not available | 0.0s | Audio not available
Kannada | kn | Audio not available | 0.0s | Audio not available
Kashmiri | ks | Audio not available | 0.0s | Audio not available
Konkani | kok | Audio not available | 0.0s | Audio not available
Maithili | mai | Audio not available | 0.0s | Audio not available
Malayalam | ml | Audio not available | 0.0s | Audio not available
Manipuri | mni | Audio not available | 0.0s | Audio not available
Marathi | mr | Audio not available | 0.0s | Audio not available
Nepali | ne | Audio not available | 0

In [9]:
ok = sum(1 for r in results if r[4] == "OK")
missing = sum(1 for r in results if r[4] == "Audio not available")

print("Languages tested:", len(results))
print("Successful transcriptions:", ok)
print("Missing audio:", missing)
print("Skipped / errors:", len(results) - ok - missing)
print("Device used:", device)

Languages tested: 22
Successful transcriptions: 0
Missing audio: 22
Skipped / errors: 0
Device used: cuda


# PART 1 STATUS: PASS

All 10 checks pass: CUDA/RTX 4050 OK, model loads on cuda, soundfile + mono + 16 kHz, one multilingual model for 22 languages, missing audio skipped, no training, no extra frameworks.

Note: only `audio/gujarati_test.wav` exists and it is empty (0 bytes), so no real transcript was produced. A valid audio file is needed to see an actual Indian-language transcript.